# ParkOpticon YOLO11s Bootstrap Pipeline
This notebook allows you to run the entire data generation, labeling, and training pipeline in an isolated run directory.
Every run creates a new, self-contained workspace so your data never gets mixed up.

The only shared assets are the reference datasets located in the project root:
- `enforcement-dataset/`
- `police-dataset-old/`
- `police-dataset-new/`


In [10]:
import os
from pathlib import Path

# ==========================================
# 1. DEFINE YOUR RUN DIRECTORY
# ==========================================
# Change this for each new run to keep your data isolated.
RUN_NAME = "Benchmark_Run_002"

# Create the isolated directory structure inside the `runs/` folder
run_dir = (Path("runs") / RUN_NAME).resolve()

if run_dir.exists():
    raise FileExistsError(
        "\n\n🛑 RUN DIRECTORY CONFLICT 🛑\n"
        f"The directory '{run_dir}' already exists!\n"
        "To prevent overwriting or mixing data, please change RUN_NAME to something new.\n"
        "If you are deliberately resuming this run, comment out this check."
    )

dirs_to_create = [
    run_dir / "manifests",
    run_dir / "lists",
    run_dir / "data" / "images_original",
    run_dir / "data" / "images_synth",
    run_dir / "data" / "labels_autogen",
    run_dir / "data" / "labels_final",
    run_dir / "data" / "splits",
    run_dir / "reports"
]

for d in dirs_to_create:
    d.mkdir(parents=True, exist_ok=True)
    
print(f"✅ Initialized isolated run directory at: {run_dir}")


FileExistsError: 

🛑 RUN DIRECTORY CONFLICT 🛑
The directory 'K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002' already exists!
To prevent overwriting or mixing data, please change RUN_NAME to something new.
If you are deliberately resuming this run, comment out this check.

## 2. Launch the Web UI & Generate Points
Run the cell below to start the Web UI. It will be bound to this specific run directory.

1. Go to **http://localhost:8000**
2. Click on **Point Manager**.
3. Draw a polygon, click **Fetch & Append** (or add points manually).
4. Click the blue **Save to Run Dir** button in the top right. This automatically places `points.csv` in your new `runs/{RUN_NAME}/manifests/` folder so you don't have to move anything manually!
5. Once saved, stop this cell (Stop button ⏹️) and continue.


In [29]:
import subprocess, signal, os

# Kill any process already using port 8000
result = subprocess.run(
    ["powershell", "-Command",
     "Get-NetTCPConnection -LocalPort 8000 -ErrorAction SilentlyContinue | Select-Object -ExpandProperty OwningProcess -Unique"],
    capture_output=True, text=True
)
for pid in result.stdout.strip().splitlines():
    pid = pid.strip()
    if pid and pid != "0":
        print(f"Killing PID {pid} on port 8000")
        os.kill(int(pid), signal.SIGTERM)

# Start the UI pointing at our isolated run directory
!python web_ui/app.py --run-dir "{run_dir}"

Killing PID 39640 on port 8000
Starting unified UI on http://0.0.0.0:8000
Using run directory: K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002


INFO:     Started server process [39640]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


^C


## 3. Fetch Street View Images
This downloads the real-world backgrounds for the locations defined in your new `points.csv`.


In [13]:
!python scripts/02_fetch_streetview.py     --points "{run_dir}"/manifests/points.csv     --manifest "{run_dir}"/manifests/images.csv     --out-dir "{run_dir}"/data/images_original



Fetching Street View:   0%|          | 0/3200 [00:00<?, ?it/s]2026-03-05 14:43:44,193 - INFO - Downloaded: K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002\data\images_original\auto_xBfhksoHNJpaAGwUghZwyw_h58\22710105e721.jpg

Fetching Street View:   0%|          | 1/3200 [00:01<1:01:21,  1.15s/it]2026-03-05 14:43:45,139 - INFO - Downloaded: K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002\data\images_original\auto_xBfhksoHNJpaAGwUghZwyw_h103\73480cfda7e6.jpg

Fetching Street View:   0%|          | 2/3200 [00:02<54:55,  1.03s/it]  2026-03-05 14:43:46,096 - INFO - Downloaded: K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002\data\images_original\auto_xBfhksoHNJpaAGwUghZwyw_h148\a9005e4579c9.jpg

Fetching Street View:   0%|          | 3/3200 [00:03<53:07,  1.00it/s]2026-03-05 14:43:47,029 - INFO - Downloaded: K:\Self Improvement\Coding\seg-4180-p

## 4. Bottom Crop
*(Optional but Recommended)* Remove the bottom 30 pixels where the Google watermark usually sits.


In [14]:
!python scripts/02b_crop_bottom.py     --manifest "{run_dir}"/manifests/images.csv     --crop-px 30


2026-03-05 15:46:00,829 - INFO - Cropped 22710105e721.jpg: removed 30px from bottom -> 22710105e721_bc30.jpg
2026-03-05 15:46:00,837 - INFO - Cropped 73480cfda7e6.jpg: removed 30px from bottom -> 73480cfda7e6_bc30.jpg
2026-03-05 15:46:00,843 - INFO - Cropped a9005e4579c9.jpg: removed 30px from bottom -> a9005e4579c9_bc30.jpg
2026-03-05 15:46:00,850 - INFO - Cropped 67741a366108.jpg: removed 30px from bottom -> 67741a366108_bc30.jpg
2026-03-05 15:46:00,856 - INFO - Cropped b693e04d097f.jpg: removed 30px from bottom -> b693e04d097f_bc30.jpg
2026-03-05 15:46:00,864 - INFO - Cropped 9f6a6732faa8.jpg: removed 30px from bottom -> 9f6a6732faa8_bc30.jpg
2026-03-05 15:46:00,871 - INFO - Cropped 96e741dc7193.jpg: removed 30px from bottom -> 96e741dc7193_bc30.jpg
2026-03-05 15:46:00,878 - INFO - Cropped d91650cfbdae.jpg: removed 30px from bottom -> d91650cfbdae_bc30.jpg
2026-03-05 15:46:00,885 - INFO - Cropped 6079fd54dfa4.jpg: removed 30px from bottom -> 6079fd54dfa4_bc30.jpg
2026-03-05 15:46:00

## 5. Detect Empty Frames
Runs YOLO to find backgrounds that have no existing vehicles. These become our blank canvases.


In [20]:
!python scripts/03_detect_empty_frames.py     --manifest "{run_dir}"/manifests/images.csv     --out-manifest "{run_dir}"/manifests/images.csv     --boxes-out "{run_dir}"/manifests/boxes_autogen.jsonl     --empty-out "{run_dir}"/lists/empty_candidates.txt     --conf 0.25     --device 0


2026-03-05 18:31:19,943 - INFO - CUDA available: NVIDIA GeForce RTX 3070 Ti
2026-03-05 18:31:19,971 - INFO - Loading YOLO model: yolo11s.pt on device: 0

Detecting vehicles: 100%|██████████| 3200/3200 [00:52<00:00, 60.90it/s]
2026-03-05 18:32:12,621 - INFO - Processed 3200 images
2026-03-05 18:32:12,621 - INFO - Found 858 empty candidates
2026-03-05 18:32:12,621 - INFO - Boxes saved to K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002\manifests\boxes_autogen_temp.jsonl
2026-03-05 18:32:12,621 - INFO - Empty candidates saved to K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002\lists\empty_candidates.txt


## 6. Synthesize Vehicle Edits
Uses the Gemini API and your reference images (police/enforcement) to generate synthetic training data inside the blank canvases.


In [23]:
!python scripts/04_synthesize_vehicle_edits.py \
    --manifest "{run_dir}"/manifests/images.csv \
    --empty-list "{run_dir}"/lists/empty_candidates.txt \
    --excluded-list "{run_dir}"/lists/excluded_from_synth.txt \
    --out-dir "{run_dir}"/data/images_synth \
    --enforcement-rate 0.35 \
    --hard-negative-rate 0.35 \
    --workers 60 \
    --seed 42


2026-03-05 19:00:05,773 - INFO - Loaded 19 reference images for enforcement
2026-03-05 19:00:05,773 - INFO - Loaded 20 reference images for police_old
2026-03-05 19:00:05,773 - INFO - Loaded 17 reference images for police_new
2026-03-05 19:00:05,793 - INFO - Creating random_vehicle for 4891bba53251 with style: gray coupe at very close (foreground, large size)
2026-03-05 19:00:05,793 - INFO - Creating random_vehicle for 17a40550c8e7 with style: green pickup truck at far away (background, small size)
2026-03-05 19:00:05,794 - INFO - Creating random_vehicle for 60990c9c42f4 with style: red pickup truck at medium distance (midground, average size)
2026-03-05 19:00:05,794 - INFO - Creating random_vehicle for a5fcb053fc7b with style: beige crossover at far away (background, small size)
2026-03-05 19:00:05,795 - INFO - Creating random_vehicle for 8a56308ca4cb with style: blue compact sedan at medium distance (midground, average size)
2026-03-05 19:00:05,795 - INFO - Creating random_vehicle fo

2026-03-05 18:38:40,193 - INFO - Loaded 19 reference images for enforcement
2026-03-05 18:38:40,193 - INFO - Loaded 20 reference images for police_old
2026-03-05 18:38:40,193 - INFO - Loaded 17 reference images for police_new
2026-03-05 18:38:40,211 - INFO - Creating random_vehicle for 4891bba53251 with style: gray coupe at very close (foreground, large size)
2026-03-05 18:38:40,211 - INFO - Creating random_vehicle for 17a40550c8e7 with style: green pickup truck at far away (background, small size)
2026-03-05 18:38:40,212 - INFO - Creating random_vehicle for 60990c9c42f4 with style: red pickup truck at medium distance (midground, average size)
2026-03-05 18:38:40,212 - INFO - Creating random_vehicle for a5fcb053fc7b with style: beige crossover at far away (background, small size)
2026-03-05 18:38:40,218 - INFO - Creating random_vehicle for 8a56308ca4cb with style: blue compact sedan at medium distance (midground, average size)

Synthesizing:   0%|          | 0/858 [00:00<?, ?it/s]2026-

## 6b. Augment Synthetic Images (Optional but Recommended)
Creates augmented copies with noise, blur, compression, and color jitter.

Helps model learn fine details and avoid overfitting to "perfect" synthetics.

In [34]:
!python scripts/04c_augment_synth_images.py \
    --manifest "{run_dir}"/manifests/images.csv \
    --synth-dir "{run_dir}"/data/images_synth \
    --copies-per-image 2 \
    --intensity extreme \
    --seed 42

2026-03-06 01:31:30,362 - INFO - Found 941 synthetic images to potentially augment

Augmenting images: 100%|██████████| 941/941 [03:55<00:00,  4.00it/s]
2026-03-06 01:35:25,803 - INFO - Created 1878 augmented images
2026-03-06 01:35:25,803 - INFO - Manifest updated: K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002\manifests\images.csv


## 7. Pre-label Synthetic Images
Auto-generates bounding boxes for the newly injected synthetic vehicles.


In [31]:
!python scripts/05_prelabel_boxes.py \
    --manifest "{run_dir}"/manifests/images.csv \
    --out-dir "{run_dir}"/data/labels_autogen \
    --conf 0.25
    

2026-03-06 00:36:58,954 - INFO - Loading YOLO model: yolo11s.pt

Pre-labeling: 100%|██████████| 6019/6019 [03:35<00:00, 27.93it/s]
2026-03-06 00:40:39,512 - INFO - Labels saved to K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002\data\labels_autogen
2026-03-06 00:40:39,514 - INFO - QA failed synthetics: 1996
2026-03-06 00:40:39,514 - INFO - Bad synthetics list: lists\bad_synthetics.txt
2026-03-06 00:40:39,514 - INFO - ============================================================
2026-03-06 00:40:39,514 - INFO - YOLO LABEL VALIDATION SUMMARY
2026-03-06 00:40:39,514 - INFO - Total boxes validated: 14436
2026-03-06 00:40:39,514 - INFO - Invalid boxes found: 0
2026-03-06 00:40:39,514 - INFO - All labels passed validation!
2026-03-06 00:40:39,514 - INFO - ============================================================


## 8. Manual Label Review (Web UI)
Launch the UI again to review the auto-generated boxes.
1. Click **Labeler** on the dashboard.
2. Drag/draw boxes to correct them.
3. When finished, stop this cell.


In [ ]:
import subprocess, signal, os

# Kill any process already using port 8000
result = subprocess.run(
    ["powershell", "-Command",
     "Get-NetTCPConnection -LocalPort 8000 -ErrorAction SilentlyContinue | Select-Object -ExpandProperty OwningProcess -Unique"],
    capture_output=True, text=True
)
for pid in result.stdout.strip().splitlines():
    pid = pid.strip()
    if pid and pid != "0":
        print(f"Killing PID {pid} on port 8000")
        os.kill(int(pid), signal.SIGTERM)

# Start the UI pointing at our isolated run directory
!python web_ui/app.py --run-dir "{run_dir}"

Starting unified UI on http://0.0.0.0:8000
Using run directory: K:\Self Improvement\Coding\seg-4180-project\parkopticon-yolo11s-bootstrap\runs\Benchmark_Run_002


INFO:     Started server process [65248]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8000): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


## 9. Split Dataset (Leakage-Safe)
Splits your reviewed data into Train, Validation, and Test sets based on Panorama IDs to prevent spatial leakage.


In [ ]:
!python scripts/06_split_dataset.py     --manifest "{run_dir}"/manifests/images.csv     --labels-dir "{run_dir}"/data/labels_autogen     --labels-final-dir "{run_dir}"/data/labels_final     --out-dir "{run_dir}"/data/splits     --train-ratio 0.75     --val-ratio 0.15     --test-ratio 0.10     --validate-leakage


## 10. Train YOLO11s
Kick off the model training process on your freshly generated and split dataset.
The weights will be saved into `runs/{RUN_NAME}/model_train/`.


In [ ]:
!python scripts/07_train_yolo.py     --data "{run_dir}"/data/splits/data.yaml     --epochs 50     --batch 16     --imgsz 640     --device 0     --name "{RUN_NAME}"_model     --project "{run_dir}"/model_train


## 11. Evaluate the Model
Run predictions on the test set and generate a final Markdown and JSON report.
*(Make sure to update the weights path to point to your `best.pt` file generated above)*


In [ ]:
# Replace 'best.pt' path below with your actual run name inside {run_dir}/model_train/
!python scripts/08_eval_report.py     --weights "{run_dir}/model_train/{RUN_NAME}_model/weights/best.pt"     --data "{run_dir}"/data/splits/data.yaml     --out-dir "{run_dir}"/reports     --conf 0.25


## 12. Advanced: Fine-Tune a Previous Model on This Run's Data
If you have a model from an older run (e.g., `parkopticon_final_v1`) and you want to continue training it against the data generated in *this* run, you simply pass the old model's `.pt` file as the `--model` argument instead of starting from scratch.

Uncomment and edit the path below to run this cross-training workflow.


In [ ]:
# PATH_TO_OLD_MODEL = "runs/detect/parkopticon_final_v1/weights/best.pt"

# !python scripts/07_train_yolo.py #     --model "{PATH_TO_OLD_MODEL}" #     --data "{run_dir}"/data/splits/data.yaml #     --epochs 30 #     --batch 16 #     --imgsz 640 #     --device 0 #     --name "{RUN_NAME}"_finetuned_from_older_model #     --project "{run_dir}"/model_train


## 13. Advanced: Evaluate a Previous Model on This Run's Test Set
Similarly, if you just want to *evaluate* (not train) an older model against this new run's test set to see how it generalizes to new locations:


In [ ]:
# PATH_TO_OLD_MODEL = "runs/detect/parkopticon_final_v1/weights/best.pt"

# !python scripts/08_eval_report.py #     --weights "{PATH_TO_OLD_MODEL}" #     --data "{run_dir}"/data/splits/data.yaml #     --out-dir "{run_dir}"/reports/cross_eval_report #     --conf 0.25
